# الفصل 43: LLM-as-a-Judge والتقييم البشري

> معظم الاستخدامات الحقيقية لـ LLMs (محادثة، كتابة إبداعية، تلخيص مفتوح) لا تملك إجابة صحيحة واحدة قابلة للمطابقة الآلية.

- **الجزء:** 9 — التقييم الشامل
- **الفصل:** 43
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 42: معايير القياس المرجعية: Benchmarks
- **الوسوم:** LLM-as-a-Judge، Pairwise، Elo، Bias

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch43-llm-as-judge.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch43-llm-as-judge.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch43-llm-as-judge.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> معظم الاستخدامات الحقيقية لـ LLMs (محادثة، كتابة إبداعية، تلخيص مفتوح) لا تملك إجابة صحيحة واحدة قابلة للمطابقة الآلية.

> **بيانات الفصل**
>
> - **الجزء:** 9 — التقييم الشامل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 42: معايير القياس المرجعية: Benchmarks


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم الفكرة الأساسية: استخدام GPT-4/Claude كحكم
- فهم التقييم المقارن (Pairwise Comparison) — أكثر موثوقية من التقييم المطلق
- فهم Chatbot Arena ونظام Elo


## 1. لماذا يهمّك هذا الموضوع؟

معظم الاستخدامات الحقيقية لـ LLMs (محادثة، كتابة إبداعية، تلخيص مفتوح) **لا** تملك إجابة صحيحة واحدة قابلة للمطابقة الآلية. هنا يصبح استخدام نموذج قوي "كحكم" (judge) هو الأداة الأكثر عملية وقابلية للتوسّع، لكنه محفوف بتحيّزات خطيرة يجب أن تعرفها وتتحكّم بها.


## 2. الخريطة الذهنية


```text
تقييم بلا إجابة مرجعية واحدة
│
├── التقييم المطلق (Absolute)
│   └── درجة 1-10 أو Likert — معايرة صعبة وتذبذب كبير
│
├── التقييم المقارن (Pairwise)
│   └── A أفضل / B أفضل / تعادل — أكثر ثباتًا
│
├── التحيّزات
│   ├── Position bias: تفضيل الموضع الأول/الثاني
│   ├── Verbosity bias: تفضيل الإجابات الطويلة
│   ├── Self-enhancement: تفضيل مخرجات نفس العائلة
│   └── Sycophancy: مجاملة المستخدم
│
├── المعايرة
│   └── مقارنة أحكام القاضي بعيّنة بشرية (Spearman)
│
└── Elo / Bradley-Terry
    └── تحويل المقارنات إلى ترتيب عالمي
```


## 3. الشرح المتدرج

### 🟢 أ) الفكرة الأساسية: استخدام GPT-4/Claude كحكم


In [ ]:
JUDGE_PROMPT_TEMPLATE = """أنت حكمٌ خبيرٌ محايد لتقييم جودة الإجابات.

السؤال المطروح على المساعد:
{question}

الإجابة المُقدَّمة من المساعد:
{response}

قيّم الإجابة وفق المعايير التالية من 1 إلى 10 لكل معيار:
1. الدقة الواقعية (Factual Accuracy)
2. الصلة بالسؤال (Relevance)
3. الوضوح والتماسك (Clarity)
4. الاكتمال (Completeness)
5. السلامة (Safety) — هل تحتوي محتوى ضارًا؟

أعطِ تقييمك بصيغة JSON فقط:
{{"accuracy": X, "relevance": X, "clarity": X, "completeness": X, "safety": X, "overall": X,
  "reasoning": "تبرير موجز"}}
"""

def llm_judge_evaluate(judge_model_client, question: str, response: str) -> dict:
    """استدعاء نموذج قوي (مثل GPT-4) للحكم على إجابة نموذج آخر"""
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, response=response)
    judge_output = judge_model_client.generate(prompt, temperature=0.0)
    import json
    return json.loads(judge_output)


### 🟡 ب) التقييم المقارن (Pairwise Comparison) — أكثر موثوقية من التقييم المطلق


In [ ]:
PAIRWISE_JUDGE_TEMPLATE = """قارن بين الإجابتين التاليتين لنفس السؤال وحدّد أيهما أفضل.

السؤال: {question}

الإجابة أ: {response_a}

الإجابة ب: {response_b}

أيّ إجابة أفضل؟ أجب بـ "أ" أو "ب" أو "تعادل" فقط، متبوعًا بتبرير موجز.
مهم جدًا: لا تتحيّز لترتيب العرض، قيّم المحتوى فقط.
"""

def pairwise_judge_with_position_swap(judge_client, question, response_a, response_b):
    """
    تقنية أساسية لتقليل تحيّز الموضع (Position Bias):
    نُشغّل المقارنة مرتين بترتيب معكوس، ونعتبر النتيجة موثوقة فقط إذا اتّفقت المرتان
    """
    result_1 = judge_client.generate(
        PAIRWISE_JUDGE_TEMPLATE.format(question=question, response_a=response_a, response_b=response_b)
    )
    result_2 = judge_client.generate(
        PAIRWISE_JUDGE_TEMPLATE.format(question=question, response_a=response_b, response_b=response_a)
    )

    # تطبيع النتيجة الثانية (لأن أ وب تبادلا الأماكن)
    swap_map = {"أ": "ب", "ب": "أ", "تعادل": "تعادل"}
    normalized_result_2 = swap_map.get(result_2.strip()[0], "غير محدد")

    if result_1.strip()[0] == normalized_result_2[0]:
        return result_1.strip()[0]   # النتيجتان متّفقتان، نثق بها
    else:
        return "غير حاسم (تحيّز موضع محتمل)"


### 🔴 ج) طبقة التعمّق: تحيّزات LLM-as-a-Judge الخطيرة ومعالجتها

| نوع التحيّز | الوصف | طريقة المعالجة |
|---|---|---|
| **Position Bias** | تفضيل الإجابة المعروضة أولاً | تبديل الترتيب وتشغيل مرتين (أعلاه) |
| **Verbosity Bias** | تفضيل الإجابات الأطول حتى لو لم تكن أفضل | إضافة تعليمة صريحة "لا تُفضّل الطول"، أو تطبيع بالطول |
| **Self-Enhancement Bias** | النموذج يُفضّل أسلوب إجاباته الخاصة | استخدام حكم من عائلة نماذج مختلفة عن النماذج المُقيَّمة |
| **Self-Consistency منخفض** | نفس المدخل يُعطي أحكامًا مختلفة بين مرات التشغيل | `temperature=0.0` + تشغيل متعدد وأخذ الأغلبية (majority voting) |


In [ ]:
def robust_judge_evaluation(judge_client, question, response, num_votes=3):
    """تقليل عدم الاتساق عبر التصويت بالأغلبية"""
    votes = []
    for _ in range(num_votes):
        result = llm_judge_evaluate(judge_client, question, response)
        votes.append(result["overall"])

    import statistics
    return {
        "median_score": statistics.median(votes),
        "std_dev": statistics.stdev(votes) if len(votes) > 1 else 0,
        "is_consistent": statistics.stdev(votes) < 1.5 if len(votes) > 1 else True
    }


> 💡 **نصيحة عملية مهمة**: حتى أفضل أنظمة LLM-as-a-Judge تحتاج **معايرة دورية** (calibration) مقابل عيّنة من التقييم البشري الحقيقي. احسب **معامل ارتباط Spearman** بين أحكام النموذج القاضي وأحكام البشر على عيّنة من 100-200 حالة؛ إذا كان الارتباط منخفضًا (<0.7)، فإن نظام القاضي الآلي غير موثوق للاعتماد عليه وحده.

### 🟡 د) Chatbot Arena ونظام Elo

**Chatbot Arena** (LMSYS) يستخدم تقييمًا بشريًا مقارنًا على نطاق واسع، محوّلاً النتائج لنظام تصنيف **Elo** (نفس نظام تصنيف الشطرنج):

$$
E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}
$$

$$
R_A' = R_A + K \cdot (S_A - E_A)
$$

حيث $R_A, R_B$ تصنيفا النموذجين، $E_A$ الاحتمال المتوقع لفوز أ، $S_A \in \{0, 0.5, 1\}$ نتيجة المقارنة الفعلية، و $K$ معامل التحديث.


In [ ]:
def update_elo_ratings(rating_a: float, rating_b: float, outcome_a: float, k_factor=32):
    """
    outcome_a: 1.0 (فوز أ) | 0.5 (تعادل) | 0.0 (فوز ب)
    """
    expected_a = 1 / (1 + 10 ** ((rating_b - rating_a) / 400))
    expected_b = 1 - expected_a

    new_rating_a = rating_a + k_factor * (outcome_a - expected_a)
    new_rating_b = rating_b + k_factor * ((1 - outcome_a) - expected_b)
    return new_rating_a, new_rating_b


## 4. مثال محلول خطوة بخطوة

لنحسب نتيجة مقارنة ثنائية بين نموذجين A وB على 20 سؤالًا، مع تبديل الموضع:

| السؤال | A أولًا | B أولًا | الحكم النهائي |
|---|---|---|---|
| 1 | A | A | A |
| 2 | B | B | B |
| 3 | A | B | تعادل (تحيّز موضع) |
| 4 | A | A | A |
| … | … | … | … |

**الحساب:** إذا حصل A على 12 فوزًا وB على 6 وتعادل 2، فإن نسبة فوز A = 12/20 = 60%. لكن قبل الاحتفال، افحص:
- كم مرة انقلب الحكم عند تبديل الموضع؟ إذا كان 4/20 = 20%، فالتقييم غير مستقر.
- هل القاضي من نفس عائلة A؟ إذا نعم، اطرح عينة معايرة بشرية.

**الخلاصة:** لا تنشر نسبة فوز واحدة دون فحص تحيّز الموضع ومعايرة بشرية.


## 5. نصائح وحيل

> 💡 **استخدم قاضيًا من عائلة مختلفة**: لا تستخدم GPT-4 لتقييم GPT-4؛ جرّب Claude أو Qwen أو نموذجًا مفتوحًا قويًا.

> 💡 **بدّل الموضع دائمًا**: كل مقارنة تُجرى مرتين (A/B وB/A)؛ إذا اختلف الحكم، سجّله كتعادل أو أعد التقييم.

> 💡 **أعطِ rubric واضحة**: معايير مثل الصحة، الالتزام بالتعليمات، الأمان، والأسلوب تقلّل التذبذب.

> 💡 **استخدم temperature=0**: القاضي يجب أن يكون حتميًا قدر الإمكان.

> 💡 **عاير مقابل البشر دوريًا**: 100-200 حالة كافية لحساب ارتباط Spearman؛ إذا <0.6، أعد ضبط الـrubric.


## 6. أخطاء شائعة ومصائد

> ⚠️ **الاعتماد على تقييم مطلق 1-10**: نفس الإجابة قد تحصل على 6 أو 9 حسب السياق. المقارنة الثنائية أكثر ثباتًا.

> ⚠️ **تحيّز الطول**: النماذج تميل لتفضيل الإجابات الأطول. أضف معيار "الإيجاز" أو قارن إجابات متساوية الطول.

> ⚠️ **Self-enhancement bias**: القاضي يفضّل مخرجات عائلته. استخدم قاضيًا مستقلًا أو عدة قضاة.

> ⚠️ **عدم معايرة القاضي**: بدون عيّنة بشرية، قد يكون القاضي متسقًا لكنه خاطئ منهجيًا.

> ⚠️ **تجاهل التكلفة**: تقييم كل زوج مرتين × 1000 سؤال مكلف. استخدم عينة ذكية أو قاضيًا أصغر.


## 7. جدول مقارنة

| المعيار | تقييم بشري | LLM-as-a-Judge | مقاييس آلية (BLEU..) |
|---|---|---|---|
| **التكلفة** | عالية جدًا | متوسطة | زهيدة جدًا |
| **السرعة** | بطيئة جدًا | سريعة | فورية |
| **الموثوقية** | الأعلى (مرجع ذهبي) | جيدة إذا عُولج التحيّز | منخفضة للمهام المفتوحة |
| **قابلية التوسّع** | محدودة جدًا | عالية | غير محدودة |
| **الاستخدام الأمثل** | التحقق النهائي، معايرة القاضي الآلي | التقييم اليومي المستمر (CI/CD) | المراقبة السريعة أثناء التطوير |


## 8. ملخص أفضل الممارسات

- استخدم LLM-as-a-Judge من **عائلة مختلفة تمامًا** عن النموذج المُقيَّم لتجنّب Self-Enhancement Bias.
- طبّق دائمًا تبديل الموضع (Position Swap) في المقارنات الثنائية.
- عايِر نظام القاضي الآلي دوريًا مقابل عيّنة بشرية (شهريًا على الأقل في الإنتاج).
- احتفظ بمجموعة اختبار بشرية صغيرة لكن عالية الجودة (Golden Set) ثابتة لا تتغيّر، لتتبّع التراجع (regression) عبر الزمن.

---


## 9. أسئلة مقابلات

**س1: لماذا يُفضَّل التقييم المقارن على المطلق؟**

**الإجابة النموذجية:** التقييم المطلق يتطلب معايرة داخلية ثابتة لما يعنيه كل رقم، وهذه المعايرة تتذبذب بين الاستدعاءات. المقارنة الثنائية تسأل سؤالًا أبسط ("أيهما أفضل؟") وتُنتج أحكامًا أكثر اتساقًا، خاصة عند تبديل الموضع.

**س2: ما هي أخطر تحيّزات LLM-as-a-Judge؟**

**الإجابة النموذجية:** تحيّز الموضع (تفضيل الأول/الثاني)، تحيّز الطول (تفضيل الإجابات الطويلة)، Self-enhancement (تفضيل مخرجات نفس العائلة)، والانحياز للمستخدم/المجاملة. تُعالج بتبديل الموضع، rubric، قضاة متعددين، ومعايرة بشرية.

**س3: كيف تعاير قاضيًا آليًا؟**

**الإجابة النموذجية:** تأخذ عيّنة 100-200 حالة، يحكمها خبراء بشريون، ثم تحسب ارتباط Spearman/Kendall بين ترتيب القاضي وترتيب البشر. إذا كان الارتباط منخفضًا، تحسّن الـrubric أو تستبدل القاضي.


## 10. تمرين عملي

1. صمّم تجربة مقارنة بين نموذجين على 20 سؤالًا، مع تبديل الموضع.
2. احسب نسبة الفوز ونسبة انقلاب الحكم بسبب الموضع.
3. أضف rubric من 3 معايير وكرّر؛ هل زاد الاتساق؟
4. قارن حكم GPT-4o بحكم Claude على 10 حالات، واحسب نسبة الاتفاق.
5. اكتب تقريرًا: هل القاضي موثوق؟ وما حدود الثقة؟


## 11. مراجع للتعمّق

- Zheng, L., et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS.
- Chiang, W.-L., et al. (2024). *Chatbot Arena: An Open Platform for Evaluating LLMs by Human Preference*.
- Liu, Y., et al. (2023). *G-Eval: NLG Evaluation using GPT-4 with Better Human Alignment*.
- Wang, Y., et al. (2023). *PandaLM: An Automatic Evaluation Benchmark for LLM Instruction Tuning Optimization*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التقييم الشامل / الفصل 43: LLM-as-a-Judge والتقييم البشري.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
